
<h1 style="text-align: center;">CIÊNCIA DE DADOS</h1>
<h1 style="text-align: center;">Roteiro de Atividade Prática</h1>
<br>
<br>

Nome: ______________________________________________________________________________________      

Turma: ______________


**Componente:** Inteligência Artificial 
<br>
**Unidade Curricular:** Projeto de Inteligência Artificial
<br>
**Tema da Semana:** Projeto de CNN
<br>


# Aula 2: Desenvolvimento do Projeto de CNN

## Tarefa
- Execute o código abaixo.

- Se as bibliotecas não tiverem instaladas, instale-as.

- Acompanhe as instruções.

- Preencha o código onde está (Adicione o seu código aqui neste bloco:)

- Responda as perguntas e conclua a atividade


### Instalação das dependências

In [ ]:
!pip install --upgrade pip setuptools wheel

In [ ]:
!pip install tensorflow

In [ ]:
!pip install  Pillow  matplotlib scipy

In [ ]:
#!conda install anaconda::tensorflow

### Descompressão das imagens

In [ ]:
!unzip -o -qq train.zip

In [ ]:
!unzip -o -qq test.zip

### Imports de dependências

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, GlobalAveragePooling2D
from tensorflow.keras.layers import Conv2D, MaxPooling2D, ZeroPadding2D
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import os, random
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.mobilenet import preprocess_input
import matplotlib.pyplot as plt


### Configurações

In [ ]:
SEED = 42


def set_seed(seed=0):
  np.random.seed(seed) 
  tf.random.set_seed(seed) 
  random.seed(seed)
  os.environ['TF_DETERMINISTIC_OPS'] = "1"
  os.environ['TF_CUDNN_DETERMINISM'] = "1"
  os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(SEED)

# Caminho das pastas com as imagens
train_data_dir = 'bird_images_sample/train'
validation_data_dir = 'bird_images_sample/test'


# Tamanho das imagens de entrada
img_rows, img_cols = 224, 224


# Quantidade de Classes
num_classes = 2

# Tamanho das amostras de treinamento e validação
nb_train_samples = 800
nb_validation_samples = 100

# Número de épocas de treinamento
epochs = 5

# Tamanho do batch size
batch_size = 16

#### Curvas de treinamento do modelo

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

# Plot da perda de treinamento e validação
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Treinamento')
plt.plot(history.history['val_loss'], label='Validação')
plt.title('Curva de Perda')
plt.xlabel('Épocas')
plt.ylabel('Perda')
plt.legend()

# Plot da acurácia de treinamento e validação
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Treinamento')
plt.plot(history.history['val_accuracy'], label='Validação')
plt.title('Curva de Acurácia')
plt.xlabel('Épocas')
plt.ylabel('Acurácia')
plt.legend()
plt.savefig('training_validation_loss_accuracy.png')

plt.show()

### Leitura dos dados

In [ ]:
# Configuração do gerador de dados para treinamento com data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,                 # Normaliza os pixels para o intervalo [0, 1]
    rotation_range=45,              # Rotaciona imagens aleatoriamente até 45 graus
    width_shift_range=0.3,          # Desloca horizontalmente até 30% da largura
    height_shift_range=0.3,         # Desloca verticalmente até 30% da altura
    horizontal_flip=True,           # Espelha horizontalmente as imagens
    fill_mode='nearest'             # Preenche os espaços vazios com o pixel mais próximo
)


validation_datagen = ImageDataGenerator(rescale=1./255)


train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_rows, img_cols),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

validation_generator = validation_datagen.flow_from_directory(
    validation_data_dir,
    target_size=(img_rows, img_cols),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

### Criação o modelo

#### Utilize o modelo MobileNet com pesos do imagenet sem o topo da Rede 

In [ ]:
# Adicione o seu código aqui neste bloco:
include_top = 
# Carrega o modelo pré-treinado
MobileNet = MobileNet(weights='imagenet', include_top=include_top, input_shape=(img_rows, img_cols, 3))

#### Congele as camadas iniciais do modelo para que não sejam treinadas

In [ ]:
# Adicione o seu código aqui neste bloco:

# Congela as camadas iniciais do model
for layer in MobileNet.layers:
    layer.trainable = 
    

### Adicione a função de ativação e o percentual para as camadas de dropout.

In [ ]:
# Adicione o seu código aqui neste bloco:

# função de ativação
hidden_activation = 

dropout_rate = 

def add_model(bottom_model, num_classes):

    top_model = bottom_model.output
    top_model = GlobalAveragePooling2D()(top_model)

    top_model = Dense(1024)(top_model)
    top_model = Activation(hidden_activation)(top_model)
    top_model = Dropout(dropout_rate)(top_model)  # Dropout 
    
    top_model = Dense(1024)(top_model)
    top_model = Activation(hidden_activation)(top_model)
    top_model = Dropout(dropout_rate)(top_model)  # Dropout 
    
    top_model = Dense(512)(top_model)
    top_model = Activation(hidden_activation)(top_model)
    top_model = Dropout(dropout_rate)(top_model)  # Dropout 
    
    top_model = Dense(num_classes, activation='softmax')(top_model)

    return top_model

In [ ]:
FC_Head = add_model(MobileNet, num_classes)

model = Model(inputs = MobileNet.input, outputs = FC_Head)

#### Implementar o Early Stopping para parar de treinar o modelo se o resultado estabilizar

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

#### Adicione o algoritmo de otimização e a função de perda (loss function) adequada para classificação multiclass (utilizamos classificação multiclasse para este caso, pois o metodo do gerador de imagens flow_from_directory esta configurado com class_mode='categorical' )

In [ ]:
# Adicione o seu código aqui neste bloco:


# algoritmo de otimização
optimizer = 

# função de perda multiclass
loss_function = 

model.compile(loss=loss_function,
              optimizer=optimizer,
              metrics=['accuracy', Precision(), Recall()])

### Treinamento do modelo

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=nb_train_samples // batch_size - 10,
    epochs=epochs,
    validation_data=validation_generator,
    validation_steps=nb_validation_samples // batch_size,
    callbacks=[early_stop],
    verbose=0
)

### Avaliação o modelo

In [ ]:
print(f'Test loss: {history.history['val_loss'][-1]:.4f}')
print(f'Test accuracy: {history.history['val_accuracy'][-1]:.4f}')
print(f'Test precision: {history.history['val_precision'][-1]:.4f}')
print(f'Test recall: {history.history['val_recall'][-1]:.4f}')

### Apresenta as predições para uma amostragem dos dados de teste

In [ ]:
# Função para carregar e preprocessar uma imagem
def load_and_preprocess_image(image_path, target_size=(224, 224)):
    img = load_img(image_path, target_size=target_size)  # Carrega a imagem com o tamanho alvo
    img_array = img_to_array(img)  # Converte a imagem para um array numpy
    img_array = preprocess_input(img_array)  # Pré-processa a imagem (para MobileNet ou modelos semelhantes)
    return img, img_array  # Retorna a imagem original e o array

# Função para prever a classe de um lote de imagens
def predict_image_class_batch(image_paths):
    images = []
    processed_images = []
    for image_path in image_paths:
        original_img, img_array = load_and_preprocess_image(image_path)
        images.append(original_img)
        processed_images.append(img_array)

    batch_array = np.stack(processed_images, axis=0)  # Cria um lote para previsão
    predictions = model.predict(batch_array, verbose=0)  # Previsão em lote
    class_labels = ['terrestres' if np.argmax(pred) == 1 else 'aquaticas' for pred in predictions]
    
    return images, class_labels  # Retorna as imagens originais e as etiquetas previstas

# Diretório contendo as imagens de teste
test_dir = 'bird_images_sample/test'


selected_items = ['Pelagic_Cormorant_0073_23785.jpg', 'Pied_Billed_Grebe_0045_35962.jpg', 'Herring_Gull_0100_46677.jpg', 'Ivory_Gull_0026_49466.jpg', 'Pied_Billed_Grebe_0007_35399.jpg', 'Heermann_Gull_0018_45608.jpg', 'Pelagic_Cormorant_0010_23711.jpg', 'Ring_Billed_Gull_0095_50362.jpg', 'Slaty_Backed_Gull_0030_796003.jpg', 'Horned_Grebe_0055_35104.jpg',
                 'Pine_Warbler_0002_171176.jpg', 'Savannah_Sparrow_0047_119365.jpg', 'Nashville_Warbler_0005_167103.jpg', 'American_Crow_0099_25717.jpg', 'American_Crow_0011_25151.jpg', 'Red_Winged_Blackbird_0020_4050.jpg', 'Purple_Finch_0025_28174.jpg', 'American_Crow_0137_25221.jpg', 'Purple_Finch_0036_27641.jpg', 'Magnolia_Warbler_0040_165921.jpg']

# Prever e exibir imagens em lotes de 5 para cada classe
for subdir in ['aquaticas', 'terrestres']:
    img_dir = os.path.join(test_dir, subdir)
    print(f"\nPredições para a classe {subdir} :")

    image_files = [f for f in os.listdir(img_dir) if f in selected_items]

    # Divide os arquivos de imagem em lotes de 5
    for i in range(0, len(image_files), 5):
        batch_files = image_files[i:i+5]
        batch_paths = [os.path.join(img_dir, img_file) for img_file in batch_files]
        
        # Obtém as previsões para o lote
        images, labels = predict_image_class_batch(batch_paths)
        
        # Exibe as imagens e previsões lado a lado
        fig, axes = plt.subplots(1, len(images), figsize=(20, 5))  # Cria uma linha única de subplots
        for ax, img, label in zip(axes, images, labels):
            ax.imshow(img)
            ax.set_title(f"Predição: {label}", fontsize=15)
            ax.axis('off')
        plt.tight_layout()
        plt.show()


### Perguntas e Conclusão

#### Altere os blocos de código com o comentário "Adicione o seu código aqui neste bloco:" e execute o notebook para ver o resultado. 


##### Quais foram as métricas da avaliação do modelo?¶

Test loss: ___  
Test accuracy: ___  
Test precision: ___  
Test recall: ___  